In [9]:
import os, sys

PROJECT_ROOT = '/scratch/jq2uw/derm_vlms'
sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
from eval.metrics import eval_model, compute_metrics

VLMS = {
    'MedGemma':     'results/medgemma_predictions_all.csv',
    'GPT-5.3':      'results/gpt53_predictions_all.csv',
    'DermatoLlama': 'results/dermato_llama_predictions_all.csv',
}

dfs = {}
metrics = {}
for name, path in VLMS.items():
    full = os.path.join(PROJECT_ROOT, path)
    if not os.path.exists(full):
        print(f'[SKIP] {name}: {path} not found')
        continue
    df = pd.read_csv(full)
    df = eval_model(df)
    dfs[name] = df
    metrics[name] = compute_metrics(df)
    print(f'{name}: {len(df)} rows loaded')

print(f'\nLoaded {len(dfs)} models')

MedGemma: 3263 rows loaded
GPT-5.3: 3263 rows loaded
DermatoLlama: 3263 rows loaded

Loaded 3 models


In [10]:
# Table 1: Overall top-1 / top-3 accuracy
rows = []
for name, m in metrics.items():
    ov = m['overall'].iloc[0]
    rows.append({
        'Model': name,
        'N': int(ov['n']),
        'Parsed (%)': f"{ov['parsed'] / ov['n'] * 100:.1f}",
        'Matched (%)': f"{ov['matched'] / ov['n'] * 100:.1f}",
        'Top-1 Acc (%)': f"{ov['top1_acc'] * 100:.1f}",
        'Top-3 Acc (%)': f"{ov['top3_acc'] * 100:.1f}",
    })
table1 = pd.DataFrame(rows)
print('=== Overall Accuracy ===')
table1

=== Overall Accuracy ===


,Model,N,Parsed (%),Matched (%),Top-1 Acc (%),Top-3 Acc (%)
0,MedGemma,3263,100.0,100.0,29.6,51.9
1,GPT-5.3,3263,97.9,97.7,27.6,49.7
2,DermatoLlama,3263,66.8,63.1,18.0,32.1


In [11]:
# Table 2: Accuracy by image_mode
frames = []
for name, m in metrics.items():
    t = m['by_image_mode'].copy()
    t.insert(0, 'Model', name)
    frames.append(t)
table2 = pd.concat(frames, ignore_index=True)
table2['top1_acc'] = (table2['top1_acc'] * 100).round(1)
table2['top3_acc'] = (table2['top3_acc'] * 100).round(1)
table2 = table2.rename(columns={'top1_acc': 'Top-1 (%)', 'top3_acc': 'Top-3 (%)', 'n': 'N'})
print('=== Accuracy by Image Mode ===')
table2

=== Accuracy by Image Mode ===


,Model,image_mode,Top-1 (%),Top-3 (%),N
0,MedGemma,dscope,32.1,56.2,1039
1,MedGemma,photo,29.2,51.3,1038
2,MedGemma,combined,31.8,55.4,1031
3,MedGemma,virtual,1.9,3.2,155
4,GPT-5.3,dscope,29.8,52.8,1039
5,GPT-5.3,photo,25.3,47.1,1038
6,GPT-5.3,combined,31.4,56.0,1031
7,GPT-5.3,virtual,1.3,3.9,155
8,DermatoLlama,dscope,21.9,36.4,1039
9,DermatoLlama,photo,11.3,22.5,1038


In [12]:
## Final Report Tables

# --- Table 1: Overall accuracy ---
report1 = table1[['Model', 'N', 'Top-1 Acc (%)', 'Top-3 Acc (%)']].copy()
report1.index = report1['Model']
report1 = report1.drop(columns='Model')
print('Overall Diagnostic Accuracy (Top3 differentials by VLMs)')
display(report1)

# --- Table 2: Accuracy by image mode per model (virtual merged into photo) ---
mode_map = {'photo': 'Clinical Photo', 'virtual': 'Clinical Photo',
            'dscope': 'Dermoscopy', 'combined': 'Combined'}
mode_order = ['Clinical Photo', 'Dermoscopy', 'Combined']
model_order = ['MedGemma', 'GPT-5.3', 'DermatoLlama']

for model_name in model_order:
    df_m = dfs[model_name].copy()
    df_m['Image Mode'] = df_m['image_mode'].map(mode_map)
    rows_b = []
    for mode in mode_order:
        grp = df_m[df_m['Image Mode'] == mode]
        rows_b.append({
            'Image Mode': mode,
            'N': len(grp),
            'Top-1 (%)': round(grp['top1_correct'].mean() * 100, 1),
            'Top-3 (%)': round(grp['top3_correct'].mean() * 100, 1),
        })
    t = pd.DataFrame(rows_b).set_index('Image Mode')
    print(f'\n{model_name} - Accuracy by Image Mode')
    display(t)

Overall Diagnostic Accuracy (Top3 differentials by VLMs)


,N,Top-1 Acc (%),Top-3 Acc (%)
Model,,,
MedGemma,3263,29.6,51.9
GPT-5.3,3263,27.6,49.7
DermatoLlama,3263,18.0,32.1



MedGemma - Accuracy by Image Mode


,N,Top-1 (%),Top-3 (%)
Image Mode,,,
Clinical Photo,1193,25.6,45.0
Dermoscopy,1039,32.1,56.2
Combined,1031,31.8,55.4



GPT-5.3 - Accuracy by Image Mode


,N,Top-1 (%),Top-3 (%)
Image Mode,,,
Clinical Photo,1193,22.2,41.5
Dermoscopy,1039,29.8,52.8
Combined,1031,31.4,56.0



DermatoLlama - Accuracy by Image Mode


,N,Top-1 (%),Top-3 (%)
Image Mode,,,
Clinical Photo,1193,10.0,19.9
Dermoscopy,1039,21.9,36.4
Combined,1031,23.2,41.8


In [13]:
# Table 3: Accuracy by y3 class
frames = []
for name, m in metrics.items():
    t = m['by_y3'].copy()
    t.insert(0, 'Model', name)
    frames.append(t)
table3 = pd.concat(frames, ignore_index=True)
table3['top1_acc'] = (table3['top1_acc'] * 100).round(1)
table3['top3_acc'] = (table3['top3_acc'] * 100).round(1)
table3 = table3.rename(columns={'top1_acc': 'Top-1 (%)', 'top3_acc': 'Top-3 (%)', 'n': 'N', 'ground_truth': 'y3'})
print('=== Accuracy by y3 Class ===')
table3

=== Accuracy by y3 Class ===


,Model,y3,Top-1 (%),Top-3 (%),N
0,MedGemma,malignant,43.4,73.5,1394
1,MedGemma,benign,27.3,50.4,1315
2,MedGemma,other,0.5,0.7,554
3,GPT-5.3,malignant,24.2,57.7,1394
4,GPT-5.3,benign,41.8,60.5,1315
5,GPT-5.3,other,2.2,4.0,554
6,DermatoLlama,malignant,17.0,31.8,1394
7,DermatoLlama,benign,26.5,45.6,1315
8,DermatoLlama,other,0.0,0.5,554


In [14]:
# Table 4: Accuracy by y16 class
frames = []
for name, m in metrics.items():
    t = m['by_y16'].copy()
    t.insert(0, 'Model', name)
    frames.append(t)
table4 = pd.concat(frames, ignore_index=True)
table4['top1_acc'] = (table4['top1_acc'] * 100).round(1)
table4['top3_acc'] = (table4['top3_acc'] * 100).round(1)
table4 = table4.rename(columns={'top1_acc': 'Top-1 (%)', 'top3_acc': 'Top-3 (%)', 'n': 'N'})
print('=== Accuracy by y16 Diagnosis ===')
table4

=== Accuracy by y16 Diagnosis ===


,Model,y16,Top-1 (%),Top-3 (%),N
0,MedGemma,Basal Cell Carcinoma,75.6,92.0,603
1,MedGemma,Melanocytic Nevus,55.7,76.5,571
2,MedGemma,Other Benign,5.9,22.2,409
3,MedGemma,Seborrheic Keratosis,3.8,51.0,239
4,MedGemma,Melanoma,39.1,79.1,230
5,MedGemma,Squamous Cell Carcinoma,16.2,90.7,204
6,MedGemma,Actinic Keratosis,14.2,56.3,183
7,MedGemma,Squamous Cell Carcinoma In Situ,0.0,0.0,159
8,MedGemma,Melanocytic Lesion,0.0,0.0,108
9,MedGemma,Dermatofibroma,2.1,4.2,48


In [15]:
# Table 5: Unmatched diagnosis terms (for refining synonym dictionary)
for name, m in metrics.items():
    um = m['unmatched']
    print(f'\n=== {name}: {len(um)} unmatched terms ===')
    if not um.empty:
        print(um['unmatched_term'].value_counts().head(20))


=== MedGemma: 54 unmatched terms ===
unmatched_term
Sebaceous Gland Carcinoma            6
Lichen Simplex Chronicus             6
Herpes Zoster                        3
Infection                            3
Insect Bites                         3
Telangiectasias                      2
Oral Candidiasis                     2
Livedo Reticularis                   2
Melasma                              2
Herpes Simplex Virus (HSV) Lesion    2
Leukoplakia                          2
Drug Eruption                        1
Vitiligo                             1
Foreign Body Granuloma               1
Miliaria                             1
Herpes Labialis                      1
Otitis Externa                       1
Erythema Nodosum                     1
Acanthosis Nigricans                 1
Vasculitis                           1
Name: count, dtype: int64

=== GPT-5.3: 268 unmatched terms ===
unmatched_term
Dilated pore of Winer                         6
Hidrocystoma                            

In [16]:
NOTEBOOK_DIR = os.path.dirname(os.path.abspath('eval_summary.ipynb'))
OUT_XLSX = os.path.join(PROJECT_ROOT, 'eval', 'notebooks', 'eval_report.xlsx')

mode_map = {'photo': 'Clinical Photo', 'virtual': 'Clinical Photo',
            'dscope': 'Dermoscopy', 'combined': 'Combined'}
mode_order = ['Clinical Photo', 'Dermoscopy', 'Combined']
model_order = ['MedGemma', 'GPT-5.3', 'DermatoLlama']

with pd.ExcelWriter(OUT_XLSX, engine='openpyxl') as writer:
    report1.to_excel(writer, sheet_name='Overall')

    for model_name in model_order:
        df_m = dfs[model_name].copy()
        df_m['Image Mode'] = df_m['image_mode'].map(mode_map)
        rows_b = []
        for mode in mode_order:
            grp = df_m[df_m['Image Mode'] == mode]
            rows_b.append({
                'Image Mode': mode,
                'N': len(grp),
                'Top-1 (%)': round(grp['top1_correct'].mean() * 100, 1),
                'Top-3 (%)': round(grp['top3_correct'].mean() * 100, 1),
            })
        pd.DataFrame(rows_b).set_index('Image Mode').to_excel(
            writer, sheet_name=model_name)

print(f'Saved 4 sheets to {OUT_XLSX}')

Saved 4 sheets to /scratch/jq2uw/derm_vlms/eval/notebooks/eval_report.xlsx
